# AI6130 Assignment 2 - FULL RUN (4-5 hours)

**Complete assignment with single GPU**

Required: 4 models x 2 benchmarks = 8 evaluations
Bonus: Optional (set ENABLE_BONUS = True/False)


In [ ]:
import os
import time
import re
import pandas as pd
from datetime import datetime

print("="*80)
print("FULL RUN - Assignment 2")
print("="*80)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# CONFIGURATION
ENABLE_BONUS = True

print(f"Config: Bonus {'Enabled' if ENABLE_BONUS else 'Disabled'}")
print()

# STEP 1: Setup
print("[1/5] Setup...")
os.chdir('/kaggle/working')

if not os.path.exists('AI6130_Assignment2'):
    os.system('git clone https://github.com/duyngtr16061999/AI6130_Assignment2')

os.chdir('AI6130_Assignment2')

os.system('pip install -q fire datasets')
os.system('pip install -q transformers==4.38.0 accelerate==0.28.0')
os.system('pip uninstall -q -y peft')
os.environ["WANDB_MODE"] = "disabled"
print("  [OK]")
print()

# STEP 2: Enable KV cache
print("[2/5] Enable KV cache...")
os.system("sed -i 's/use_cache=False/use_cache=True/g' evaluate.py")
print("  [OK]")
print()

# STEP 3: Clean models
print("[3/5] Clean models...")
if os.path.exists('./trained_models'):
    import shutil
    shutil.rmtree('./trained_models')
os.makedirs('./trained_models', exist_ok=True)
print("  [OK]")
print()

# STEP 4: TRAINING & EVALUATION (Single GPU)
print("="*80)
print("[4/5] TRAINING & EVALUATION (Single GPU)")
print("="*80)
print()

models = [
    ("openlm-research/open_llama_3b_v2", "OpenLLaMA"),
    ("TinyLlama/TinyLlama_v1.1", "TinyLlama")
]

adapters = [
    ("lora", "", "LoRA", "LoRA"),
    ("bottleneck", "--use_adapterp", "AdapterP", "AdapterP")
]

datasets_required = ["AddSub", "AQuA"]
datasets_bonus = ["MultiArith", "SingleEq", "gsm8k", "SVAMP"]

all_results = []

def extract_accuracy(log_file):
    try:
        with open(log_file, 'r') as f:
            text = f.read()
        match = re.search(r'accuracy\s+(\d+)\s+([\d.]+)', text, re.IGNORECASE)
        if match:
            return float(match.group(2)) * 100
    except:
        pass
    return None

# Required experiments
for model_path, model_name in models:
    for (adapter_name, extra_flags, adapter_display, eval_name) in adapters:
        out_dir = f"./trained_models/{model_name}-{adapter_display}"

        print(f"{'='*60}")
        print(f"TRAIN: {model_name} + {adapter_display}")
        print(f"{'='*60}")

        # IMPORTANT: CUDA_VISIBLE_DEVICES=0 forces single GPU!
        train_cmd = (
            f"CUDA_VISIBLE_DEVICES=0 python finetune.py --base_model '{model_path}' "
            f"--data_path './ft-training_set/math_7k.json' "
            f"--output_dir '{out_dir}' --batch_size 4 --micro_batch_size 4 "
            f"--num_epochs 1 --learning_rate 3e-4 --cutoff_len 256 "
            f"--val_set_size 120 --adapter_name {adapter_name} {extra_flags}"
        )

        start = time.time()
        result = os.system(train_cmd)
        elapsed = time.time() - start

        print(f"\n[{'OK' if result == 0 else 'FAIL'}] Training ({elapsed/60:.1f} min)\n")

        if result == 0:
            for dataset in datasets_required:
                print(f"  Eval: {dataset}")

                log_file = f"/tmp/eval_{model_name}_{adapter_display}_{dataset}.txt"
                eval_cmd = (
                    f"CUDA_VISIBLE_DEVICES=0 python evaluate.py --adapter {eval_name} "
                    f"--dataset {dataset} --base_model '{model_path}' "
                    f"--lora_weights '{out_dir}' > {log_file} 2>&1"
                )

                os.system(eval_cmd)
                acc = extract_accuracy(log_file)

                all_results.append({
                    'Model': f"{model_name}-{adapter_display}",
                    'Dataset': dataset,
                    'Accuracy': acc if acc else 'N/A',
                    'Type': 'Required'
                })

                if acc:
                    print(f"    [OK] {acc:.2f}%")
                else:
                    print(f"    [WARN] Could not parse")
        print()

# Bonus
if ENABLE_BONUS:
    print("="*80)
    print("BONUS EXPERIMENTS")
    print("="*80)
    print()

    model_path = "TinyLlama/TinyLlama_v1.1"
    model_name = "TinyLlama"
    adapter_name = "lora"
    extra_flags = ""
    adapter_display = "LoRA"
    eval_name = "LoRA"
    out_dir = "./trained_models/TinyLlama-LoRA-bonus"

    print(f"TRAIN: {model_name} + {adapter_display} (3 epochs, math_10k)")

    train_cmd = (
        f"CUDA_VISIBLE_DEVICES=0 python finetune.py --base_model '{model_path}' "
        f"--data_path './ft-training_set/math_10k.json' "
        f"--output_dir '{out_dir}' --batch_size 4 --micro_batch_size 4 "
        f"--num_epochs 3 --learning_rate 3e-4 --cutoff_len 256 "
        f"--val_set_size 120 --adapter_name {adapter_name} {extra_flags}"
    )

    start = time.time()
    result = os.system(train_cmd)
    elapsed = time.time() - start

    print(f"\n[{'OK' if result == 0 else 'FAIL'}] Training ({elapsed/60:.1f} min)\n")

    if result == 0:
        all_datasets = datasets_required + datasets_bonus
        for dataset in all_datasets:
            print(f"  Eval: {dataset}")

            log_file = f"/tmp/eval_bonus_{dataset}.txt"
            eval_cmd = (
                f"CUDA_VISIBLE_DEVICES=0 python evaluate.py --adapter {eval_name} "
                f"--dataset {dataset} --base_model '{model_path}' "
                f"--lora_weights '{out_dir}' > {log_file} 2>&1"
            )

            os.system(eval_cmd)
            acc = extract_accuracy(log_file)

            all_results.append({
                'Model': f"{model_name}-{adapter_display}-BONUS",
                'Dataset': dataset,
                'Accuracy': acc if acc else 'N/A',
                'Type': 'Bonus'
            })

            if acc:
                print(f"    [OK] {acc:.2f}%")
            else:
                print(f"    [WARN] Could not parse")
    print()

# STEP 5: RESULTS
print()
print("="*80)
print("[5/5] FINAL RESULTS")
print("="*80)
print()

results_df = pd.DataFrame(all_results)

results_df.to_csv('assignment2_results.csv', index=False)
print("[OK] Saved: assignment2_results.csv")

pivot = results_df[results_df['Type'] == 'Required'].pivot_table(
    index='Model',
    columns='Dataset',
    values='Accuracy',
    aggfunc='first'
)

print()
print("REQUIRED RESULTS:")
print(pivot.to_string())
print()

pivot.to_csv('assignment2_results_pivot.csv')
print("[OK] Saved: assignment2_results_pivot.csv")

if ENABLE_BONUS and any(r['Type'] == 'Bonus' for r in all_results):
    print()
    print("BONUS RESULTS:")
    bonus_df = results_df[results_df['Type'] == 'Bonus']
    print(bonus_df[['Model', 'Dataset', 'Accuracy']].to_string(index=False))

print()
print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
print()
print("[OK] ALL DONE! Download CSV files.")
